# Make the photo's patterns stronger for LoFTR (iteration 34)

**Idea (user).** The patterns can be seen by eye in the thermal photo, so the breakthrough should come from the photo side: keep the features or make them stronger before LoFTR.

**Why the pipeline loses them (measured).** At 1000-2000 m the canvas pixel (level 10-11: 9.6 or 13.6 m) is 1.4-1.9x bigger than the photo pixel (5-10 m), and `prepare_photo` blurs the photo to match. Clumps of 1-3 photo pixels are averaged away before LoFTR sees them, and LoFTR then works on a 28 x 28 grid of 8-pixel patches.

**Variants (LoFTR at the true place, 6 queries, 142 development photos above 800 m; control = a wrong tile 3 km away).**
- `base`: the pipeline.
- `cleaned`: vignette removal + local normalisation 15 px on the photo, local normalisation matched on the tile (iteration 33).
- `cleaned + denoise + boost`: as cleaned, then non-local-means denoising and a band-pass boost (photo + 1.5 x (Gaussian 1 px - Gaussian 2.5 px)) so the clumps are stronger.
- `native`: the canvas drawn at TWO LEVELS FINER (448 x 448 px, pixel 2x smaller, same footprint) with the map tile at the same finer level, so the photo keeps its own resolution (the photo pixel is now about the canvas pixel) and the clumps are 2x bigger in pixels for LoFTR.
- `native + cleaned (+ tile LN)` and `native + cleaned + denoise + boost (+ tile LN)`.

**Kill rule (written before the run).** A variant is useful only if the share of photos with at least 8 true-place inliers above 800 m rises by at least 10 points over `base` (40 %), with the share of at least 8 inliers at the wrong place staying 0, and the gain is not confined to one height band. Development photos only; true-place diagnostic, not the full pipeline.

In [ ]:
import os, sys, math, colorsys
import numpy as np, pandas as pd, torch, cv2
HEADLESS = os.environ.get("EF_HEADLESS", "../../../../Main_Pipeline_Headless"); sys.path.insert(0, os.path.abspath(HEADLESS))
import settings
settings.LEVELS = [int(v) for v in os.environ.get("EF_LEVELS", "3,4,5,6,7,8,9,10,11").split(",")]
from geoloc import photo, queries, scoring, fine_matching, flat_field
from geoloc.cli import read_rows, move_hint
from geoloc.coarse_search import DEVICE, score_map_for_query, pick_peaks, window_transforms, zncc_everywhere, fast_length, cut_map_features
from geoloc.edge_features import normalise_cells
from geoloc.levels import level_for_height, level_gsd, map_piece
from geoloc.map_store import MapStore
CELL = settings.CELL; TILE = settings.TILE
DATASET = os.environ["EF_DATASET"]; STORE = os.environ["EF_STORE"]; OUT = os.environ.get("EF_OUT", "results/edge_features"); os.makedirs(OUT, exist_ok=True)
store = MapStore(STORE); rows = read_rows(DATASET)       # the dataset folder holds the development photos only
heights = np.array([float(r["alt"]) for r in rows]); print(len(rows), "photos, heights", heights.min(), heights.max())

In [ ]:
import matplotlib; matplotlib.use("Agg")
loftr = fine_matching.load_loftr(os.path.join(os.environ.get("EF_MODELS", os.path.join(os.path.abspath(HEADLESS), "models")), settings.LOFTR_FILE))
ORIGINAL_NORMALISE = photo.normalise_frame
def with_settings(flat=None, degree=3, local=None):
    settings.FLAT_FIELD_FROM_HEIGHT_M = flat; settings.FLAT_FIELD_DEGREE = degree; settings.LOCAL_NORM_SIGMA = local
def clean_tile(tile, sigma):
    has = tile > 0; fill = float(tile[has].mean()) if has.any() else 0.5
    normal = flat_field.local_normalise(np.where(has, tile, fill).astype(np.float32), sigma); out = ORIGINAL_NORMALISE(normal)
    return np.where(has, out, float(out[has].mean()) if has.any() else 0.5).astype(np.float32)
def denoise_boost(img, denoise=True, boost=True):
    out = img.astype(np.float32)
    if denoise:
        u8 = np.clip(out * 255, 0, 255).astype(np.uint8); out = cv2.fastNlMeansDenoising(u8, None, h=8, templateWindowSize=5, searchWindowSize=15).astype(np.float32) / 255
    if boost:
        out = out + 1.5 * (cv2.GaussianBlur(out, (0, 0), 1.0) - cv2.GaussianBlur(out, (0, 0), 2.5)); out = np.clip(out, 0, 1)
    return out
def best_inliers(canvases, tile):
    best = 0
    for canvas_picture in canvases:
        pa, pb, conf = fine_matching.loftr_matches(loftr, canvas_picture, tile); keep = conf >= settings.LOFTR_CONFIDENCE
        matrix, inl = fine_matching.fit_similarity(pa[keep], pb[keep]); best = max(best, int(inl.sum()) if matrix is not None else 0)
    return best
def wrong_tile(level_image, level, truth, size):
    for dx, dy in ((3000.0, 0.0), (-3000.0, 0.0), (0.0, 3000.0), (0.0, -3000.0)):
        cand = map_piece(level_image, level, truth[0] + dx, truth[1] + dy, size)
        if (cand > 0).mean() > 0.9: return cand
    return None
# variant: (resolution drop in levels, photo cleaned, denoise+boost, tile LN)
VARIANTS = [("base", 0, False, False, False), ("cleaned", 0, True, False, True), ("cleaned + denoise + boost", 0, True, True, True),
            ("native 448", 2, False, False, False), ("native 448 + cleaned", 2, True, False, True), ("native 448 + cleaned + denoise + boost", 2, True, True, True)]
query_list = queries.query_list(); records = []
for index, meta in enumerate(rows):
    if float(meta["alt"]) < 800: continue
    camera = photo.read_camera(meta); pose = photo.read_pose(meta); path = os.path.join(DATASET, "images", meta["filename"]); level = level_for_height(pose["height"], camera)
    truth = store.gps_to_metres(float(meta["lat_gtruth"]), float(meta["lon_gtruth"])); photo_gsd = pose["height"] / camera["fx"]
    for name, drop, clean, enhance, tile_ln in VARIANTS:
        lf = level - drop
        if lf < min(settings.LEVELS) or lf not in store.levels: continue
        size = settings.TILE * (2 ** (drop // 2)); settings.TILE = size
        try:
            if clean: with_settings(0, 3, 15.0)
            else: with_settings(None)
            picture = photo.prepare_photo(path, pose, camera, lf); with_settings(None)
            if enhance: picture = denoise_boost(picture)
            canvases = [fine_matching.grey_canvas(*photo.draw_canvas(picture, pose, camera, lf, q["heading"], q["scale_factor"])) for q in query_list]
            true_tile = map_piece(store.levels[lf], lf, truth[0], truth[1], size); bad_tile = wrong_tile(store.levels[lf], lf, truth, size)
            if bad_tile is None: continue
            if tile_ln:
                sigma_tile = max(3.0, 15.0 * photo_gsd / level_gsd(lf)); true_tile = clean_tile(true_tile, sigma_tile); bad_tile = clean_tile(bad_tile, sigma_tile)
            records.append(dict(photo=index, height=pose["height"], variant=name, true_inl=best_inliers(canvases, true_tile), wrong_inl=best_inliers(canvases, bad_tile)))
        finally:
            settings.TILE = 224
    if index % 20 == 0: print(index, flush=True)
tab = pd.DataFrame(records); tab.to_csv(os.path.join(OUT, "loftr_photo_strength.csv"), index=False)
tab["band"] = pd.cut(tab.height, [800, 1100, 1400, 2100]); order = [v[0] for v in VARIANTS]
summ = lambda g: pd.Series(dict(n=len(g), true_ge8=(g.true_inl >= 8).mean(), true_ge4=(g.true_inl >= 4).mean(), true_median=g.true_inl.median(), wrong_ge8=(g.wrong_inl >= 8).mean(), wrong_ge4=(g.wrong_inl >= 4).mean()))
res = tab.groupby(["band", "variant"], observed=True).apply(summ)
for col in ["true_ge8", "true_ge4", "wrong_ge8", "wrong_ge4", "true_median"]:
    print(col); print(res[col].unstack("variant")[[o for o in order if o in res[col].unstack("variant").columns]].round(2).to_string())
print("above 800 m"); print(tab.groupby("variant").apply(summ).loc[[o for o in order if o in tab.variant.unique()]].round(3).to_string())